In [ ]:
import pandas as pd
from dateutil import parser

def flexible_date_parser(date_str):
    try:
        return parser.parse(date_str)
    except Exception:
        return pd.NaT
def combine_datasets(file_list, start_date, end_date, output_file):
    dfs = []
    col_names_seen = set()
    for i, file in enumerate(file_list):
        df = pd.read_csv(file)
        df['Date'] = df['Date'].astype(str).str.strip().apply(flexible_date_parser)
        df['Date'] = df['Date'].dt.normalize()
        df = df.dropna(subset=['Date'])
        df = df.sort_values('Date')
        df = df.set_index('Date')
        df = df[~df.index.duplicated(keep='first')]
        new_columns = []
        for col in df.columns:
            new_col = f"{col}_file{i+1}"
            count = 1
            while new_col in col_names_seen:
                count += 1
                new_col = f"{col}_file{i+1}_{count}"
            col_names_seen.add(new_col)
            new_columns.append(new_col)
        df.columns = new_columns
        dfs.append(df)
    combined_df = pd.concat(dfs, axis=1)
    combined_df = pd.concat(dfs, axis=1)
    combined_df.index = pd.to_datetime(combined_df.index)
    combined_df = combined_df[~combined_df.index.duplicated(keep='first')]
    business_days = pd.bdate_range(start=start_date, end=end_date)
    combined_df = combined_df.reindex(business_days).ffill()
    combined_df = combined_df.reset_index().rename(columns={'index': 'Date'})
    combined_df.to_csv(output_file, index=False)
    print(f"Combined file saved as {output_file}")
file_list = ['EBL UG Historical Data.csv','KES to UGX Shilling.csv','KCB UG Historical Data.csv','KNCB KN Historical Data.csv']
start_date = '2017-05-15'              
end_date = '2025-08-19'               
output_file = 'combined_data_EBL_KCBUG_KCBKY.csv' 
combine_datasets(file_list, start_date, end_date, output_file)
#######
